# 05 — Selección trimestral de componentes (IEC15) · metodología v0.4

Aplica las reglas de la sección 3 de la metodología a las 90 candidatas, en cada revisión trimestral desde noviembre de 2019 hasta mayo de 2026:

1. **Historia mínima:** 6 meses de cotización a la fecha de referencia.
2. **Frecuencia de negociación:** % de días hábiles de los últimos 3 meses con al menos una transacción. Mínimo 80% para nuevas y 70% para vigentes (umbrales de MSCI).
3. **Liquidez:** ranking por mediana del valor diario transado (MDVT) de los últimos 6 meses.
4. **Una serie por empresa:** entra solo la más líquida (SQM-A/SQM-B, ANDINA-A/ANDINA-B).
5. **Colchón:** las 12 primeras entran directo; los 3 cupos restantes, primero para vigentes dentro de las 18 primeras.

La sección de **validación** documenta por qué la versión 0.4 reemplazó la presencia bursátil (1.000 UF) por la frecuencia de negociación.

**Limitación conocida:** AES Andes y Grupo Security no tienen datos en Yahoo (notebook 04), así que no participan en la selección.

Antes de hacer commit: **Edit → Clear Outputs of All Cells**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import bcchapi

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CARPETA_RAW = RAIZ / "data" / "raw"
CARPETA_PROC = RAIZ / "data" / "processed"
CARPETA_PROC.mkdir(parents=True, exist_ok=True)

token = (RAIZ / "credenciales.txt").read_text(encoding="utf-8-sig").strip()
siete = bcchapi.Siete(token=token)
print("Listo")

## 1. Universo y calendario de días hábiles

Un día cuenta como hábil si el IPSA tuvo cierre (datos diarios de Investing).

In [ ]:
universo = pd.read_csv(RAIZ / "data" / "reference" / "universo_candidatos.csv")
cand = universo[universo["decision"] == "Candidata"]
TICKERS = cand["simbolo"].tolist()
GRUPO = cand.set_index("simbolo")["grupo_empresa"]
print(len(TICKERS), "candidatas")

ipsa_raw = pd.read_csv(CARPETA_RAW / "ipsa_investing.csv", thousands=",")
CAL = pd.DatetimeIndex(pd.to_datetime(ipsa_raw.iloc[:, 0], format="%m/%d/%Y")).sort_values()
CAL = CAL[CAL <= "2026-07-15"]
print("Calendario:", CAL.min().date(), "→", CAL.max().date(), "|", len(CAL), "días hábiles")

## 2. Descargar precios y volúmenes

Puede tardar uno o dos minutos. Los tickers que fallen en la descarga conjunta se reintentan uno por uno.

In [ ]:
INICIO, FIN = "2019-01-01", "2026-07-16"
datos = yf.download(TICKERS, start=INICIO, end=FIN, auto_adjust=False, progress=False, threads=True)

def limpiar_indice(df):
    df = df.copy()
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df.index = df.index.normalize()
    return df

CLOSE = limpiar_indice(datos["Close"]).reindex(CAL)
VOL = limpiar_indice(datos["Volume"]).reindex(CAL)

def sin_datos():
    return [t for t in TICKERS if t not in CLOSE.columns or CLOSE[t].isna().all()]

for t in sin_datos():
    try:
        h = yf.Ticker(t).history(start=INICIO, end=FIN, auto_adjust=False)
    except Exception:
        continue
    if not h.empty:
        h = limpiar_indice(h)
        CLOSE[t] = h["Close"].reindex(CAL)
        VOL[t] = h["Volume"].reindex(CAL)

fallidos = sin_datos()
print("Con datos:", len(TICKERS) - len(fallidos), "| Sin datos:", len(fallidos), fallidos)

CLOSE.to_csv(CARPETA_RAW / "close_candidatos.csv")
VOL.to_csv(CARPETA_RAW / "volumen_candidatos.csv")

## 3. UF diaria (Banco Central)

Ya no se usa para seleccionar; solo para la sección de validación, que compara contra la medida anterior de presencia bursátil (1.000 UF).

In [ ]:
uf_opciones = siete.buscar("Unidad de fomento")
uf_opciones = uf_opciones[uf_opciones["frequencyCode"] == "DAILY"]
CODIGO_UF = "F073.UFF.PRE.Z.D"
if CODIGO_UF not in set(uf_opciones["seriesId"]):
    CODIGO_UF = uf_opciones["seriesId"].iloc[0]

uf_df = siete.cuadro(series=[CODIGO_UF], nombres=["uf"], desde=INICIO, hasta=FIN)
uf_df.index = pd.to_datetime(uf_df.index)
UF = uf_df["uf"].astype(float).reindex(CAL).ffill().bfill()
print("Serie usada:", CODIGO_UF)
UF.loc[["2019-11-15", "2023-11-30", "2026-05-15"]]

## 4. Reglas de selección

Funciones probadas con datos simulados antes de entregarse.

In [ ]:
import numpy as np
import pandas as pd

N_COMPONENTES, N_DIRECTAS, TOP_VIGENTES = 15, 12, 18
FREC_NUEVA, FREC_VIGENTE = 80, 70          # frecuencia de negociación mínima (%), umbrales MSCI
MESES_FRECUENCIA, MESES_LIQUIDEZ = 3, 6
MINIMO = 12

def tercer_viernes(anio, mes):
    d = pd.Timestamp(anio, mes, 1)
    return d + pd.Timedelta(days=(4 - d.weekday()) % 7 + 14)

def dia_habil_previo(fecha, CAL):
    pos = CAL.searchsorted(fecha, side="right") - 1
    return CAL[pos]

def ventana(CAL, fecha_ref, meses):
    return CAL[(CAL > fecha_ref - pd.DateOffset(months=meses)) & (CAL <= fecha_ref)]

def metricas(fecha_ref, CAL, VOL, VALOR_CLP, PRIMERA, GRUPO):
    v_frec = ventana(CAL, fecha_ref, MESES_FRECUENCIA)
    v_liq = ventana(CAL, fecha_ref, MESES_LIQUIDEZ)
    frecuencia = (VOL.loc[v_frec] > 0).sum() / len(v_frec) * 100
    mdvt = VALOR_CLP.loc[v_liq].median()
    return pd.DataFrame({
        "frecuencia": frecuencia,
        "mdvt_mm": mdvt / 1e6,
        "historia_ok": PRIMERA.notna() & (PRIMERA <= fecha_ref - pd.DateOffset(months=MESES_LIQUIDEZ)),
        "grupo": GRUPO,
    })

def seleccionar(m, vigentes):
    umbral = np.where(m.index.isin(list(vigentes)), FREC_VIGENTE, FREC_NUEVA)
    eleg = m[m["historia_ok"] & (m["frecuencia"] >= umbral)].copy()
    eleg = eleg.sort_values("mdvt_mm", ascending=False)
    eleg = eleg[~eleg["grupo"].duplicated(keep="first")]      # una serie por empresa
    eleg["ranking"] = range(1, len(eleg) + 1)
    directas = list(eleg.index[:N_DIRECTAS])
    cupos = N_COMPONENTES - len(directas)
    vigentes_colchon = [t for t in eleg.index[N_DIRECTAS:TOP_VIGENTES] if t in vigentes]
    resto = [t for t in eleg.index[N_DIRECTAS:] if t not in vigentes_colchon]
    seleccion = directas + (vigentes_colchon + resto)[:max(cupos, 0)]
    return seleccion, eleg


## 5. Valor transado diario

Valor = precio de cierre × volumen. Un día sin transacciones vale 0; un día antes de que la acción cotizara queda vacío.

In [ ]:
VALOR_CLP = (CLOSE * VOL.fillna(0)).where(CLOSE.notna())
VALOR_UF = VALOR_CLP.div(UF, axis=0)
PRIMERA = pd.to_datetime(CLOSE.apply(lambda s: s.first_valid_index()))
GRUPO = GRUPO.reindex(CLOSE.columns)
print("Matriz de valor transado:", VALOR_CLP.shape)

## 6. Validación: por qué frecuencia de negociación y no presencia bursátil

**Prueba A — casos oficiales.** IAM entró al IPSA en septiembre de 2019 (cumplía 90% de presencia oficial en agosto). SalfaCorp e ILC eran componentes vigentes (mínimo 85%). La aproximación con 1.000 UF y volumen de Yahoo les da mucho menos; la frecuencia de negociación los reconoce.

**Prueba B — consistencia del volumen de Yahoo.** Se compara la suma del valor transado de Yahoo con una serie mensual del Banco Central. El nivel no calza (la serie oficial mide algo más acotado), pero la razón entre ambas es estable hasta 2023 y salta desde 2024: el nivel del volumen de Yahoo no es comparable entre años, y un umbral en pesos o UF no es confiable.

In [ ]:
f_ref = dia_habil_previo(pd.Timestamp("2019-08-16"), CAL)
pos = CAL.get_loc(f_ref)
v180 = CAL[max(0, pos - 179): pos + 1]
casos = ["IAM.SN", "SALFACORP.SN", "ILC.SN"]
m_ago19 = metricas(f_ref, CAL, VOL, VALOR_CLP, PRIMERA, GRUPO)

prueba_a = pd.DataFrame({
    "presencia_aprox_1000UF": ((VALOR_UF.loc[v180, casos] >= 1000).sum() / 180 * 100).round(1),
    "frecuencia_3m": m_ago19.loc[casos, "frecuencia"].round(1),
    "minimo_oficial_presencia": [90, 85, 85],
})
display(prueba_a)

m_nov19 = metricas(dia_habil_previo(pd.Timestamp("2019-11-15"), CAL), CAL, VOL, VALOR_CLP, PRIMERA, GRUPO)
print((m_nov19["frecuencia"] >= FREC_NUEVA).sum(), "acciones con frecuencia de 80% o más en noviembre de 2019")

In [ ]:
op = siete.buscar("Monto transacción")
fila = op[op["spanishTitle"].str.contains("Santiago") & op["spanishTitle"].str.contains("Millones de pesos")]
COD_MONTO = fila["seriesId"].iloc[0].strip()

oficial = siete.cuadro(series=[COD_MONTO], nombres=["oficial_mm"], desde=INICIO, hasta="2026-07-01")
oficial.index = pd.to_datetime(oficial.index)
yahoo_mes = (VALOR_CLP.sum(axis=1) / 1e6).resample("MS").sum().rename("yahoo_mm")
prueba_b = pd.concat([yahoo_mes, oficial["oficial_mm"].astype(float)], axis=1, join="inner")
print("Serie oficial:", COD_MONTO)
prueba_b.resample("YE").sum().assign(razon_pct=lambda d: d.yahoo_mm / d.oficial_mm * 100).round(0)

## 7. Revisiones trimestrales

La primera revisión (referencia noviembre 2019) define la composición inicial del 31-dic-2019. La última es la de mayo de 2026. La de agosto de 2026 no se ejecuta por falta de datos (sección 6 de la metodología).

In [ ]:
revisiones, filas_comp, vigentes = [], [], []
corto = lambda xs: ", ".join(sorted(x.replace(".SN", "") for x in xs))

for anio in range(2019, 2027):
    for mes_ref, mes_ef in [(2, 3), (5, 6), (8, 9), (11, 12)]:
        ref = tercer_viernes(anio, mes_ref)
        if ref < pd.Timestamp("2019-11-01") or ref > pd.Timestamp("2026-05-31"):
            continue
        fecha_ref = dia_habil_previo(ref, CAL)
        inicial = (anio, mes_ref) == (2019, 11)
        fecha_ef = pd.Timestamp("2019-12-31") if inicial else tercer_viernes(anio, mes_ef)

        m = metricas(fecha_ref, CAL, VOL, VALOR_CLP, PRIMERA, GRUPO)
        sel, eleg = seleccionar(m, vigentes)

        revisiones.append({
            "fecha_referencia": fecha_ref.date(),
            "fecha_efectiva": fecha_ef.date(),
            "elegibles": len(eleg),
            "componentes": len(sel),
            "entran": "(composición inicial)" if inicial else corto(set(sel) - set(vigentes)),
            "salen": "" if inicial else corto(set(vigentes) - set(sel)),
        })
        for t in sel:
            filas_comp.append({
                "fecha_referencia": fecha_ref.date(),
                "fecha_efectiva": fecha_ef.date(),
                "ticker": t,
                "ranking": int(eleg.at[t, "ranking"]),
                "mdvt_mm_clp": round(eleg.at[t, "mdvt_mm"], 1),
                "frecuencia_pct": round(eleg.at[t, "frecuencia"], 1),
            })
        vigentes = sel

resumen = pd.DataFrame(revisiones)
comp = pd.DataFrame(filas_comp)
print("Revisiones:", len(resumen), "| Bajo el mínimo de", MINIMO, ":", (resumen["componentes"] < MINIMO).sum())
resumen

## 8. Composición inicial (31-dic-2019)

In [ ]:
inicial = comp[comp["fecha_efectiva"] == pd.Timestamp("2019-12-31").date()]
inicial.sort_values("ranking").reset_index(drop=True)

## 9. Guardar

- `data/processed/composiciones.csv` y `resumen_revisiones.csv`: solo fechas, tickers y ranking. **Se pueden publicar.**
- `data/raw/metricas_seleccion.csv`: incluye MDVT y frecuencia calculadas con datos de Yahoo. **No se publica.**

In [ ]:
comp[["fecha_referencia", "fecha_efectiva", "ticker", "ranking"]].to_csv(
    CARPETA_PROC / "composiciones.csv", index=False)
comp.to_csv(CARPETA_RAW / "metricas_seleccion.csv", index=False)
resumen.to_csv(CARPETA_PROC / "resumen_revisiones.csv", index=False)
print("Guardado.")

## Qué revisar y enviar

1. La tabla de la **prueba A** y el conteo de noviembre de 2019 (sección 6).
2. La tabla **resumen de revisiones** (sección 7).
3. La **composición inicial** (sección 8).